# 크롤링을 활용한 데이터 수집 — 노트북 개요

웹페이지에서 원하는 정보를 **자동으로 가져오는 방법**을 기초부터 실전까지 단계별로 연습하는 노트북입니다.

| 용어 | 뜻 | 사용 도구 |
|---|---|---|
| **크롤링(Crawling)** | 웹 서버에 요청해서 페이지(HTML)를 **받아오는** 것 | `requests`, `urllib` |
| **스크래핑(Scraping)** | 받아온 HTML에서 원하는 부분만 **골라내는** 것 | `BeautifulSoup` |

**진행 순서**
1. **requests** — 웹페이지 요청, 상태 코드(200/404/500)로 성공·실패 확인
2. **urllib** — 같은 일을 파이썬 기본 모듈로 (오류를 `try/except`로 처리)
3. **BeautifulSoup** — 연습용 HTML에서 `find` / `findAll` / class / 속성(`href`)으로 원하는 태그 꺼내기
4. **실전: 네이버 검색** — 검색 결과 페이지를 받아와 링크 목록 추출
5. **CSS 선택자** — `select` / `select_one`으로 원하는 요소만 골라내기
6. **CSV 저장** — 추출한 결과를 pandas 표로 만들어 파일로 저장

## 01.Request 
* Crawling 라이브러리

In [2]:
# 네이버 연결 확인
import requests  # HTTP 요청을 보내는 외부 라이브러리 (크롤링의 기본 도구)

# GET 요청: 해당 주소의 웹페이지를 달라고 서버에 요청 → 응답(Response) 객체가 res에 담김
res = requests.get("http://naver.com")
# status_code: 서버가 돌려준 HTTP 상태 코드 (200=성공, 404=페이지 없음, 500=서버 오류 등)
print("status_code : ", res.status_code)  # 정상 : 200

status_code :  200


In [3]:
# 연결 에러 처리
    # 4로 시작하는 error : client error  (예: 404 없는 페이지, 403 접근 금지)
    # 5로 시작하는 error : server error  (예: 500 서버 내부 오류)
# 존재하지 않는(또는 막힌) 블로그 주소로 요청 → 200이 아닌 코드가 나오는지 확인
# requests는 4xx/5xx가 와도 예외를 내지 않고 status_code에 담아서 돌려줌
res = requests.get("http://yyy.tistory.com")
print("status_code : ", res.status_code)

status_code :  404


In [4]:
# 연결 에러 처리
res = requests.get("http://yyy.tistory.com")

# 상태 코드를 직접 비교해서 성공/실패를 분기 처리하는 방법
if res.status_code == requests.codes.ok:  # requests.codes.ok = 200 (숫자 200 대신 이름으로 쓴 상수)
   print("정상")
else:
   print("오류 : ", res.status_code)  # 실패 시 어떤 코드인지 출력

오류 :  404


In [5]:
import requests

res = requests.get("http://naver.com")
# raise_for_status(): 상태 코드가 4xx/5xx면 HTTPError 예외를 발생시킴
# → if문으로 일일이 검사하지 않아도 실패 시 바로 멈추게 하는 간단한 방법
res.raise_for_status()  # error면 프로그램 종료, 성공이면 다음라인 실행

print("정상입니다.")
print(len(res.text))  # res.text: 응답 본문(HTML)을 문자열로 받은 것 → 글자 수 출력
print(res.text)       # HTML 원문 전체 출력 (매우 길다)

정상입니다.
216331
 <!doctype html> <html lang="ko" class="fzoom"> <head> <meta charset="utf-8"> <meta name="Referrer" content="origin"> <meta http-equiv="X-UA-Compatible" content="IE=edge"> <meta name="viewport" content="width=1190"> <title>NAVER</title> <meta name="apple-mobile-web-app-title" content="NAVER"/> <meta name="robots" content="index,nofollow"/> <meta name="description" content="네이버 메인에서 다양한 정보와 유용한 컨텐츠를 만나 보세요"/> <meta property="og:title" content="네이버"> <meta property="og:url" content="https://www.naver.com/"> <meta property="og:image" content="https://s.pstatic.net/static/www/mobile/edit/2016/0705/mobile_212852414260.png"> <meta property="og:description" content="네이버 메인에서 다양한 정보와 유용한 컨텐츠를 만나 보세요"/> <meta name="twitter:card" content="summary"> <meta name="twitter:title" content=""> <meta name="twitter:url" content="https://www.naver.com/"> <meta name="twitter:image" content="https://s.pstatic.net/static/www/mobile/edit/2016/0705/mobile_212852414260.png"> <meta name="twitter:de

## 02.Urllib : 크롤링 + 스크래핑

In [6]:
# urllib: 파이썬에 기본 내장된 URL 처리 모듈 (설치 불필요, requests보다 사용법이 조금 불편)
from urllib.request import urlopen

url = "https://www.naver.com"
html = urlopen(url)  # URL을 열어 응답 객체를 받음 (requests.get과 비슷한 역할)
# read(): 응답 본문을 bytes(b'...') 형태로 읽음 → 한글이 \xec... 처럼 깨져 보이는 이유
# 문자열로 보려면 html.read().decode("utf-8") 처럼 디코딩 필요
print(html.read())

b' <!doctype html> <html lang="ko" class="fzoom"> <head> <meta charset="utf-8"> <meta name="Referrer" content="origin"> <meta http-equiv="X-UA-Compatible" content="IE=edge"> <meta name="viewport" content="width=1190"> <title>NAVER</title> <meta name="apple-mobile-web-app-title" content="NAVER"/> <meta name="robots" content="index,nofollow"/> <meta name="description" content="\xeb\x84\xa4\xec\x9d\xb4\xeb\xb2\x84 \xeb\xa9\x94\xec\x9d\xb8\xec\x97\x90\xec\x84\x9c \xeb\x8b\xa4\xec\x96\x91\xed\x95\x9c \xec\xa0\x95\xeb\xb3\xb4\xec\x99\x80 \xec\x9c\xa0\xec\x9a\xa9\xed\x95\x9c \xec\xbb\xa8\xed\x85\x90\xec\xb8\xa0\xeb\xa5\xbc \xeb\xa7\x8c\xeb\x82\x98 \xeb\xb3\xb4\xec\x84\xb8\xec\x9a\x94"/> <meta property="og:title" content="\xeb\x84\xa4\xec\x9d\xb4\xeb\xb2\x84"> <meta property="og:url" content="https://www.naver.com/"> <meta property="og:image" content="https://s.pstatic.net/static/www/mobile/edit/2016/0705/mobile_212852414260.png"> <meta property="og:description" content="\xeb\x84\xa4\xec\x9d\x

In [7]:
from urllib.request import urlopen
from urllib.request import HTTPError  # 서버에는 접속됐지만 4xx/5xx 응답이 온 경우의 예외

# urllib은 requests와 달리 4xx/5xx 응답이면 바로 예외를 발생시킴 → try/except로 처리
try:
    html = urlopen("http://www.google.com/kim.html")  # 구글에 없는 페이지 → 404 발생
except HTTPError as e:
    print(e)        # 예: HTTP Error 404: Not Found
else:
    print("성공")   # 예외가 없을 때만 실행되는 블록


HTTP Error 404: Not Found


In [8]:
# errorURL
from urllib.request import urlopen
from urllib.request import HTTPError  # 서버는 있는데 페이지 요청이 실패한 경우 (404 등)
from urllib.request import URLError   # 서버 자체를 찾을 수 없는 경우 (도메인 없음, 네트워크 끊김 등)

try:
    html = urlopen("http://www.dddsdf.com/kim.html")  # 존재하지 않는 도메인
except HTTPError as e:
    print(e)
except URLError as e:
    # HTTPError는 URLError의 하위 클래스라서, HTTPError를 먼저 잡아야 둘을 구분할 수 있음
    print("The server coult not be found!")
else:
    print("성공")

The server coult not be found!


## 03. BeautifulSoup : 스크래핑(scraping)

In [9]:
# BeautifulSoup 설치 명령 (이미 설치되어 있으면 실행할 필요 없음)
# conda install bs4
# !pip install bs4   ← 노트북 셀에서 ! 를 붙이면 터미널 명령으로 실행됨

In [10]:
import bs4  # BeautifulSoup: HTML 문자열을 분석(파싱)해서 원하는 태그를 쉽게 꺼내 주는 라이브러리

# 연습용 HTML (실제 웹페이지 대신 문자열로 직접 작성)
html_str = """<html>
                <div>hello</div>
            </html>"""

# HTML 문자열을 파싱해서 BeautifulSoup 객체로 만듦 ("html.parser"는 파이썬 기본 파서)
bs_obj = bs4.BeautifulSoup(html_str, "html.parser")

print(type(bs_obj))             # <class 'bs4.BeautifulSoup'>
print(bs_obj)                   # 파싱된 HTML 전체
print(bs_obj.find("div"))       # find(): 조건에 맞는 첫 번째 태그 하나 → <div>hello</div>
print(bs_obj.find("div").text)  # .text: 태그를 뺀 안쪽 글자만 → hello

<class 'bs4.BeautifulSoup'>
<html>
<div>hello</div>
</html>
<div>hello</div>
hello


In [11]:
import bs4

# <ul>(목록) 안에 <li>(항목) 3개가 있는 연습용 HTML
html_str = """
<html>
    <body>
        <ul>
            <li>hello</li>
            <li>bye</li>
            <li>welcome</li>
        </ul>
    </body>
</html>
"""

# Mission : <ul> 추출
bs_obj = bs4.BeautifulSoup(html_str, "html.parser")
ul = bs_obj.find("ul")  # 첫 번째 <ul> 태그(안의 <li>들 포함)를 통째로 가져옴
print(ul)

<ul>
<li>hello</li>
<li>bye</li>
<li>welcome</li>
</ul>


In [12]:
# <li>hello</li> 추출 & 텍스트 hello만 추출
# 첫번째 <li> 추출
li = ul.find("li")  # 문서 전체가 아니라 위에서 찾은 ul 안에서만 검색 (태그 객체에도 find 사용 가능)
print(li)           # <li>hello</li>
print(li.text)      # hello


<li>hello</li>
hello


In [13]:
# findAll(): 조건에 맞는 태그를 "전부" 찾아 리스트로 반환 (find_all과 같은 함수)
lis = ul.findAll("li")
print(lis)          # [<li>hello</li>, <li>bye</li>, <li>welcome</li>]
print(lis[1])       # 리스트이므로 인덱스로 접근 → 두 번째 항목 <li>bye</li>
print(lis[1].text)  # bye


[<li>hello</li>, <li>bye</li>, <li>welcome</li>]
<li>bye</li>
bye


C:\Users\hooho\AppData\Local\Temp\ipykernel_2800\1075656841.py:2: DeprecationWarning: Call to deprecated method findAll. (Replaced by find_all) -- Deprecated since version 4.0.0.
  lis = ul.findAll("li")


In [14]:
import bs4

# class 속성이 다른 <ul>이 2개 있는 HTML
html_str = """
<html>
    <body>
        <ul class = "greet">
            <li>hello</li>
            <li>bye</li>
            <li>welcome</li>
        </ul>
        <ul class = "reply">
            <li>ok</li>
            <li>no</li>
            <li>sure</li>
        </ul>
    </body>
</html>
"""

bs_obj = bs4.BeautifulSoup(html_str, "html.parser")
ul = bs_obj.find("ul")  # find는 첫 번째 것만 → class="greet" 인 ul만 나옴
print(ul)

<ul class="greet">
<li>hello</li>
<li>bye</li>
<li>welcome</li>
</ul>


In [15]:
# findAll로 <ul> 2개를 모두 리스트로 가져옴
uls = bs_obj.findAll("ul")
print(uls)

[<ul class="greet">
<li>hello</li>
<li>bye</li>
<li>welcome</li>
</ul>, <ul class="reply">
<li>ok</li>
<li>no</li>
<li>sure</li>
</ul>]


C:\Users\hooho\AppData\Local\Temp\ipykernel_2800\67138444.py:2: DeprecationWarning: Call to deprecated method findAll. (Replaced by find_all) -- Deprecated since version 4.0.0.
  uls = bs_obj.findAll("ul")


In [16]:
# class로 데이터 찾기
bs_obj = bs4.BeautifulSoup(html_str, "html.parser")
# 두 번째 인자로 속성 조건을 딕셔너리로 전달 → class가 "reply"인 ul만 찾음
# (같은 의미: bs_obj.find("ul", class_="reply")  ← class는 파이썬 예약어라 뒤에 _ 붙임)
ul = bs_obj.find("ul", {"class": "reply"})
print(ul)

<ul class="reply">
<li>ok</li>
<li>no</li>
<li>sure</li>
</ul>


In [17]:
import bs4

# <a> 태그(링크)가 들어 있는 HTML. href 속성에 이동할 주소가 들어 있음
html_str = """
<html>
    <body>
        <ul class = "ko">
            <li>
                <a href = "http://www.naver.com/">네이버</a>
            </li>
            <li>
                <a href = "http://www.daum.com/">다음</a>
            </li>
        </ul>
        <ul class = "sns">
            <li>
                <a href = "http://www.google.com/">구글</a>
            </li>
            <li>
                <a href = "http://www.facebook.com/">페이스북</a>
            </li>
        </ul>
    </body>
</html>
"""

bs_obj = bs4.BeautifulSoup(html_str, "html.parser")
atag = bs_obj.find("a")  # 첫 번째 <a> 태그 (네이버)
print(atag)
print(atag["href"])      # 태그["속성명"]: 속성 값 꺼내기 → http://www.naver.com/

<a href="http://www.naver.com/">네이버</a>
http://www.naver.com/


In [18]:
# 마지막 주소 추출
atags = bs_obj.findAll("a")  # <a> 태그 4개 전부 리스트로
print(atags)
print()  # 빈 줄 출력 (보기 좋게 구분용)

print(atags[-1]["href"])  # [-1]: 리스트의 마지막 요소 → 페이스북 주소


[<a href="http://www.naver.com/">네이버</a>, <a href="http://www.daum.com/">다음</a>, <a href="http://www.google.com/">구글</a>, <a href="http://www.facebook.com/">페이스북</a>]

http://www.facebook.com/


C:\Users\hooho\AppData\Local\Temp\ipykernel_2800\874589856.py:2: DeprecationWarning: Call to deprecated method findAll. (Replaced by find_all) -- Deprecated since version 4.0.0.
  atags = bs_obj.findAll("a")  # <a> 태그 4개 전부 리스트로


In [19]:
type(atags)    # bs4.element.ResultSet: findAll 결과 타입 (리스트처럼 쓰면 됨)
atags[0].text  # 첫 번째 링크의 글자 → '네이버'
# 노트북은 셀의 "마지막 줄" 값만 화면에 표시하므로 type(atags) 결과는 안 보임

'네이버'

In [20]:
# text만 추출(전체 URL)
# 모든 <a> 태그를 하나씩 꺼내 태그 전체와 글자를 출력
for x in atags:
    print(x)        # <a href="...">네이버</a>
    print(x.text)   # 네이버
    # 주소만 원하면: print(x["href"])

<a href="http://www.naver.com/">네이버</a>
네이버
<a href="http://www.daum.com/">다음</a>
다음
<a href="http://www.google.com/">구글</a>
구글
<a href="http://www.facebook.com/">페이스북</a>
페이스북


## 04. 실전 — 네이버 검색 결과 가져오기

지금까지 배운 것을 합쳐 실제 사이트에 적용합니다.
- **requests**로 "인공지능" 검색 결과 페이지를 받아오고 (브라우저인 척하는 `User-Agent` 헤더 포함)
- **BeautifulSoup**으로 파싱한 뒤, 페이지 안의 모든 링크(`<a>`)에서 **제목 + 주소**를 뽑아냅니다.

> 참고: 네이버는 검색 결과 일부를 JavaScript로 그리고 class 이름도 자주 바뀌어서, 결과 개수나 구조가 실행할 때마다 달라질 수 있습니다.

In [21]:
# ===== 실전: 네이버 검색 결과 페이지 가져오기 (requests로 크롤링 + BeautifulSoup으로 스크래핑) =====
import requests
from bs4 import BeautifulSoup

query = "인공지능"  # 검색어
# f-string으로 검색어를 URL의 query= 부분에 끼워 넣음 (나머지 파라미터는 네이버 검색 기본값)
url = f"https://search.naver.com/search.naver?where=nexearch&sm=top_hty&fbm=0&ie=utf8&query={query}&ackey=74jg70sa"

# HTTP 요청 헤더 설정
# 헤더 없이 요청하면 파이썬 프로그램(봇)으로 인식되어 차단되거나 다른 페이지를 받을 수 있음
# → 일반 크롬 브라우저인 것처럼 User-Agent를 지정
headers = {
    # 웹 서버에 HTTP 요청을 보낼 때 브라우저 및 실행 환경을 나타내는 User-Agent 정보를 지정
    "User-Agent": (
        # Windows 운영체제와 64비트 환경에 관한 정보를 나타냄
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        # 괄호 안에서 문자열을 나란히 쓰면 자동으로 하나로 이어붙여짐
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/130.0.0.0 Safari/537.36"
    )
}

# 서버에 요청 전송
response = requests.get(
    url,
    headers=headers,  # 위에서 만든 헤더를 함께 보냄
    timeout=10)  # 최대 10초까지 기다림, 시간을 넘어가면 requests.exceptions.Timeout 예외 발생

# 응답 상태 확인
print("상태 코드:", response.status_code)
print("최종 URL:", response.url)  # 리다이렉트(자동 이동)가 있었다면 실제로 도착한 주소

# HTTP 요청이 실패하면 예외 발생
response.raise_for_status()

# HTML 파싱
soup = BeautifulSoup(response.text, "html.parser")

# HTML 제목 출력
# soup.title.text
# soup.title: <title> 태그 / get_text(strip=True): 앞뒤 공백 제거한 글자
# <title>이 없으면 soup.title이 None이라 에러가 나므로, 조건부 표현식으로 "제목 없음" 출력
print("페이지 제목:", soup.title.get_text(strip=True) if soup.title else "제목 없음")

# HTML 일부 출력
print(soup.prettify()[:2000])   # HTML을 들여쓰기하여 보기 좋게 출력 ([:2000] → 앞 2000글자만)

# print(response.text)  # 가공 전 HTML 원문 전체 (너무 길어서 주석 처리)

상태 코드: 200
최종 URL: https://search.naver.com/search.naver?where=nexearch&sm=top_hty&fbm=0&ie=utf8&query=%EC%9D%B8%EA%B3%B5%EC%A7%80%EB%8A%A5&ackey=74jg70sa
페이지 제목: 인공지능 : 네이버 검색
<!DOCTYPE html>
<html lang="ko">
 <head>
  <meta charset="utf-8"/>
  <meta content="strict-origin-when-cross-origin" name="referrer"/>
  <meta content="telephone=no,address=no,email=no" name="format-detection"/>
  <meta content="인공지능 : 네이버 검색" property="og:title"/>
  <meta content="https://ssl.pstatic.net/sstatic/search/common/og_v3.png" property="og:image"/>
  <meta content="'인공지능'의 네이버 검색 결과입니다." property="og:description"/>
  <meta content="'인공지능'의 네이버 검색 결과입니다." lang="ko" name="description"/>
  <title>
   인공지능 : 네이버 검색
  </title>
  <link href="https://ssl.pstatic.net/sstatic/search/favicon/favicon_32x32_240820.ico" rel="shortcut icon"/>
  <link href="https://ssl.pstatic.net/sstatic/search/opensearch-description.https.xml" rel="search" title="Naver" type="application/opensearchdescription+xml"/>
  <script>
   

In [22]:
# 페이지에 존재하는 모든 링크 요소 검색
    # select() : css 선택자를 사용해서 HTML요소를 검색
    #   - 결과는 항상 "리스트" (하나도 없으면 빈 리스트 [])
    #   - findAll("a") 와 같은 결과지만, "ul.reply > li" 처럼 CSS 문법을 그대로 쓸 수 있어 더 많이 쓰임
    # soup은 위 셀에서 받아온 네이버 검색 결과 페이지 전체

links = soup.select("a")  # 페이지 안의 <a>(링크) 태그 전부
print(f"검색어 : {query}")
print(f"발견한 링크 수:{len(links)}")  # len(리스트) = 개수


검색어 : 인공지능
발견한 링크 수:478


In [23]:
# 주소만 추출
# 위에서 찾은 링크 태그들을 하나씩 꺼내 "글자(제목)"와 "주소(href)"를 뽑아냄
for link in links:
    # print(type(link))
    # print(link["href"])  #href가 없으면 system error
    #   → 태그["속성"] 방식은 속성이 없으면 KeyError로 멈춤
    title = link.get_text(" ", strip=True)  # 태그 안 글자. " " : 안쪽 태그 사이를 공백으로 이어붙임 / strip=True : 앞뒤 공백 제거
    href = link.get("href") # href가 없으면 None  → .get()은 에러 대신 None을 돌려줘서 안전


    # 제목과 링크가 모두 존재하는 경우만 출력
    #   - title : 빈 문자열("")이면 False → 글자 없는 링크(아이콘 등) 제외
    #   - href  : None이면 False → 주소 없는 링크 제외
    #   - href.startswith("http") : "#", "javascript:..." 같은 내부 동작용 링크 제외
    #   (and는 앞이 False면 뒤를 검사하지 않아서, href가 None일 때 startswith 에러가 나지 않음)
    if title and href and href.startswith("http"):
        print("제목:", title)
        print("링크:", href)
        print("=" * 60)  # 구분선 ("=" 60개)






제목: NAVER
링크: https://www.naver.com
제목: 어학사전 새 창 열림
링크: https://dict.naver.com/dict.search?query=%EC%9D%B8%EA%B3%B5%EC%A7%80%EB%8A%A5&from=tsearch
제목: 쇼핑 새 창 열림
링크: https://search.shopping.naver.com/search/all?where=all&frm=NVSCTAB&query=%EC%9D%B8%EA%B3%B5%EC%A7%80%EB%8A%A5
제목: 지도 새 창 열림
링크: https://map.naver.com/v5/search/%EC%9D%B8%EA%B3%B5%EC%A7%80%EB%8A%A5
제목: 검색옵션 가이드 새 창 열림
링크: https://help.naver.com/support/alias/search/integration/intergration_84.naver
제목: 도움말
링크: https://ads.naver.com/help/faq/1011?t=1778046083628
제목: 열린재정
링크: https://ader.naver.com/v1/6KLh1-dVZOWQgQmaDKA4bhkOSTiJvEoeJhxCUjuwmpmewUykREgVCxUB5-QM2_ChFIYkJEmDtH1YNYUDOsHV9Py3CHykrKSQsuDP_fd4S5-YOe_6XUCsnFjtWuLrHYrkuxyv7s-AzWvV-rMDt63adm-N9WB_QoDf-GNU47xkQwrDj94UqVAndSuhzhFDI8ljtSXQ7p1lEub0gZDf1LFUsL8-LxNWKWQ5XNJ_rdX1nNQMgqZK74Er0q3LYyy3QenG2Grbmt2vPup_U8aANQH6BE-ioKlJSui4lh6j9oi-CBXzlYEIM8sOtXFw0pKiwSvTbmWImiCPHbE2qYhEJ9BGZMFa3rsQJi5JeMsw1Vy-Vvz3fZlcYZ8lhxlAFciBY-lSomxo8SIWMsePD3C8EePdo23YapxpJuw5e1HH3RMNf0IzJbUOd

# CSS 선택자로 원하는 요소만 추출하기

CSS 선택자 기본 문법 (`select` / `select_one`에 그대로 사용)

| 선택자 | 뜻 | 예 |
|---|---|---|
| `a` | 태그 이름 | 모든 `<a>` |
| `.description` | class | `class="description"` |
| `#main` | id | `id="main"` |
| `h2 a` | 띄어쓰기 = 안쪽(자손) | `<h2>` 안의 `<a>` |
| `ul > li` | `>` = 바로 아래 자식 | `<ul>` 바로 아래 `<li>` |
| `a[href]` | 속성 | `href` 속성이 있는 `<a>` |

- `select()` → 조건에 맞는 것 **전부**를 리스트로 (= `findAll`)
- `select_one()` → **첫 번째 하나**만, 없으면 `None` (= `find`)

In [24]:
from bs4 import BeautifulSoup

# 실제 네이버 페이지는 너무 복잡하므로, 검색 결과 "한 건"의 구조를 흉내 낸 작은 HTML로 연습
#   .search-result  : 검색 결과 한 건을 감싸는 상자
#   h2 > a          : 제목 + 링크
#   p.description   : 설명 문구
html = '''
<div class="search-result">
    <h2>
        <a href="https://example.com/ai">
            인공지능 기술 동향
        </a>
    </h2>

    <p class="description">
        최근 인공지능 기술의 발전과 산업 적용 사례를 소개합니다.
    </p>
</div>
'''

In [25]:
# ⚠ 여기서 soup 변수가 연습용 HTML로 덮어써짐 → 네이버 페이지 soup은 사라짐 (그래서 아래에서 다시 요청함)
soup = BeautifulSoup(html, "html.parser")

# 검색 결과 영역 선택
# select_one() : CSS 선택자에 맞는 "첫 번째 하나"만 반환 (find와 비슷). 없으면 None
#   ".search-result" → 점(.)은 class를 뜻함 → class="search-result" 인 요소
result = soup.select_one(".search-result")
result  # 셀의 마지막 줄이라 값이 화면에 표시됨

<div class="search-result">
<h2>
<a href="https://example.com/ai">
            인공지능 기술 동향
        </a>
</h2>
<p class="description">
        최근 인공지능 기술의 발전과 산업 적용 사례를 소개합니다.
    </p>
</div>

In [26]:
# 제목과 링크 추출
# "h2 a" : 띄어쓰기 = "안에 있는" → <h2> 안에 있는 <a> 태그
#   result.select_one(...) 처럼 문서 전체가 아니라 result 상자 안에서만 찾음
title_tag = result.select_one("h2 a")

title = title_tag.get_text(" ", strip=True)   # 글자는 title에 따로 저장
link = title_tag.get("href")                   # title_tag는 그대로 태그라서 .get() 가능

#설명 추출
# ".description" → class="description" 인 요소 (<p class="description">)
description_tag = result.select_one(".description")
description = description_tag.get_text(strip=True)

print("제목:", title)
print("링크:", link)
print("설명 :", description)

제목: 인공지능 기술 동향
링크: https://example.com/ai
설명 : 최근 인공지능 기술의 발전과 산업 적용 사례를 소개합니다.


## 05. 검색 결과를 CSV 파일로 저장

1. (위 연습 셀에서 `soup`이 연습용 HTML로 바뀌었으므로) 네이버 검색 페이지를 **다시 요청**
2. 모든 링크에서 제목·주소를 뽑아 리스트에 모으고 → **pandas DataFrame(표)** 으로 변환
3. 중복 링크 제거 후 `naver_search_results.csv`로 저장 (`utf-8-sig` → 엑셀에서 한글 안 깨짐)
4. 마지막 셀: 개발자도구에서 복사한 **CSS 선택자**로 `<meta>` 태그 정보 꺼내기

In [27]:
# ===== 네이버 검색 결과 페이지 다시 가져오기 =====
# 위 연습 셀에서 soup이 연습용 HTML로 바뀌었기 때문에, CSV 저장 전에 네이버 페이지를 다시 받아 soup을 되돌려 놓는 셀
# (내용은 앞의 "실전" 셀과 같음. 출력 부분만 빠짐)
import requests
from bs4 import BeautifulSoup

query = "인공지능"  # 검색어
# f-string으로 검색어를 URL의 query= 부분에 끼워 넣음 (나머지 파라미터는 네이버 검색 기본값)
url = f"https://search.naver.com/search.naver?where=nexearch&sm=top_hty&fbm=0&ie=utf8&query={query}&ackey=74jg70sa"

# HTTP 요청 헤더 설정
# 헤더 없이 요청하면 파이썬 프로그램(봇)으로 인식되어 차단되거나 다른 페이지를 받을 수 있음
# → 일반 크롬 브라우저인 것처럼 User-Agent를 지정
headers = {
    # 웹 서버에 HTTP 요청을 보낼 때 브라우저 및 실행 환경을 나타내는 User-Agent 정보를 지정
    "User-Agent": (
        # Windows 운영체제와 64비트 환경에 관한 정보를 나타냄
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        # 괄호 안에서 문자열을 나란히 쓰면 자동으로 하나로 이어붙여짐
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/130.0.0.0 Safari/537.36"
    )
}

# 서버에 요청 전송
response = requests.get(
    url,
    headers=headers,  # 위에서 만든 헤더를 함께 보냄
    timeout=10)  # 최대 10초까지 기다림, 시간을 넘어가면 requests.exceptions.Timeout 예외 발생

# 응답 상태 확인
print("상태 코드:", response.status_code)
print("최종 URL:", response.url)  # 리다이렉트(자동 이동)가 있었다면 실제로 도착한 주소

# HTTP 요청이 실패하면 예외 발생
response.raise_for_status()

# HTML 파싱
soup = BeautifulSoup(response.text, "html.parser")

상태 코드: 200
최종 URL: https://search.naver.com/search.naver?where=nexearch&sm=top_hty&fbm=0&ie=utf8&query=%EC%9D%B8%EA%B3%B5%EC%A7%80%EB%8A%A5&ackey=74jg70sa


In [28]:
import pandas as pd  # 표(DataFrame) 형태로 데이터를 다루고 CSV/엑셀로 저장하는 라이브러리

#네이버 검색 결과를 CSV 파일로 저장
results=[]  # 링크 하나당 딕셔너리 1개씩 모을 리스트 → 나중에 표의 "행"이 됨

for link_tag in soup.select("a"):
    title = link_tag.get_text(" ",strip=True)
    href = link_tag.get("href","")  # href가 없으면 None 대신 빈 문자열 "" → 아래 startswith에서 에러 안 남


    # 글자가 있고 http로 시작하는 진짜 링크만 저장
    if title and href.startswith("http"):
        results.append({          # 딕셔너리의 키가 표의 "열 이름"이 됨
            "검색어" :query,
            "제목": title,
            "링크":href
        })

# for문이 끝난 뒤(모든 링크를 다 모은 뒤)에 한 번만 표 생성·중복 제거·저장·출력
df=pd.DataFrame(results)  # 딕셔너리 리스트 → 표

# 중복 링크 제거
if not df.empty:  # 표가 비어 있지 않을 때만
    df = df.drop_duplicates(subset=["링크"])  # "링크" 열 값이 같은 행은 첫 번째만 남김


df.to_csv(
    "naver_search_results.csv",  # 노트북과 같은 폴더에 저장
    index=False,                 # 0,1,2… 행 번호 열은 저장하지 않음
    encoding="utf-8-sig"         # 엑셀에서 열어도 한글이 안 깨지는 인코딩 (BOM 포함 UTF-8)
)

print(df.head(10))  # 앞 10행 미리보기
print("저장완료: naver_search_results.csv")

     검색어               제목                                                 링크
0   인공지능            NAVER                              https://www.naver.com
1   인공지능      어학사전 새 창 열림  https://dict.naver.com/dict.search?query=%EC%9...
2   인공지능        쇼핑 새 창 열림  https://search.shopping.naver.com/search/all?w...
3   인공지능        지도 새 창 열림  https://map.naver.com/v5/search/%EC%9D%B8%EA%B...
4   인공지능  검색옵션 가이드 새 창 열림  https://help.naver.com/support/alias/search/in...
5   인공지능              도움말  https://ads.naver.com/help/faq/1011?t=17780460...
6   인공지능             열린재정  https://ader.naver.com/v1/6KLh1-dVZOWQgQmaDKA4...
10  인공지능           열린질문AI  https://ader.naver.com/v1/d8YRFxoWXPFLuZiWEuSx...
11  인공지능            수혜서비스  https://ader.naver.com/v1/2tRuRraw_1TwTQpLlFuu...
12  인공지능           재정상세통계  https://ader.naver.com/v1/k3xWM7JO-2pBl_YBn4C9...
저장완료: naver_search_results.csv


In [29]:
# 브라우저 개발자도구(F12) → 요소 우클릭 → Copy → Copy selector 로 복사한 선택자를 그대로 사용한 예
#   "head > meta:nth-child(4)" = <head> 바로 아래 자식 중 4번째 요소이면서 <meta> 태그인 것
#   ⚠ 순서(번호)에 의존하므로 사이트 구조가 조금만 바뀌어도 다른 태그를 가리키거나 못 찾음
#     → 실무에서는 'meta[property="og:title"]' 처럼 속성으로 찾는 편이 안전
result = soup.select("head > meta:nth-child(4)") # selector 주소
print(result)
print(type(result),len(result))   # select는 항상 리스트(ResultSet) → 개수 확인
print(result[0]['content'])       # 첫 번째 meta 태그의 content 속성 값 (못 찾으면 IndexError)

[<meta content="인공지능 : 네이버 검색" property="og:title"/>]
<class 'bs4.element.ResultSet'> 1
인공지능 : 네이버 검색


## 06. 네이버 날씨 — 원하는 영역만 골라내기

네이버에서 "날씨"를 검색했을 때 가운데 나오는 **현재 날씨 박스**만 추출합니다.

| 화면에 보이는 것 | HTML 태그 | 사용한 선택자 |
|---|---|---|
| 15.2° | `<div class="temperature_text">` | `.temperature_text` |
| 어제보다 4.4° ↑ / 맑음 | `<p class="summary">` | `p.summary` |
| 체감 / 습도 / 동풍 | `<dl class="summary_list">` 안의 `<dt>`(이름) + `<dd>`(값) | `dl.summary_list div.sort` |
| 위 전부를 감싼 박스 | `<div class="weather_info">` | `div.weather_info` |

**방법:** 개발자도구(F12)에서 원하는 부분을 클릭 → 그 태그의 `class` 이름 확인 → `select` / `select_one`에 넣기

> 앞 셀처럼 "Copy selector"로 복사한 `nth-child(4)` 같은 **순서 번호** 주소보다, `class` 이름으로 찾는 쪽이 사이트가 조금 바뀌어도 잘 버팁니다.

In [30]:
import requests                 # 웹페이지를 받아오는 도구 (크롤링)
from bs4 import BeautifulSoup   # 받아온 HTML에서 원하는 부분을 골라내는 도구 (스크래핑)

# 네이버에서 "날씨"를 검색한 결과 페이지 주소 (query= 뒤가 검색어)
url = "https://search.naver.com/search.naver?query=날씨"
# 브라우저인 척하는 자기소개(User-Agent) — 내 크롬에서 navigator.userAgent 로 확인한 값
#   없으면 python-requests 로 보여서 봇으로 막히거나 다른 페이지를 받을 수 있음
headers = {"User-Agent": 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/155.0.0.0 Safari/537.36'}
# 한 줄에 3단계: ① requests.get으로 페이지 요청 → ② .text로 HTML 글자 꺼내기 → ③ BeautifulSoup으로 파싱
#   timeout=10 : 서버가 10초 안에 응답 없으면 포기(에러)  /  "html.parser" : 파이썬 기본 HTML 해석기
#   ⚠ 요청 결과를 res 같은 변수에 따로 저장하지 않았음 → 날씨 페이지는 soup 에만 들어 있음
soup = BeautifulSoup(requests.get(url, headers=headers, timeout=10).text, "html.parser")

# 1) 현재 온도 — 화면에 안 보이는 "현재 온도" 글자(시각장애인용)가 섞여 있어서 지워줌
#   .temperature_text : class="temperature_text" 인 요소 (큰 글씨 15.2° 부분)
#   select_one : 맞는 것 중 첫 번째 하나만 (없으면 None)
#   get_text(strip=True) → "현재 온도15.2°" → replace로 "현재 온도"를 빈 글자로 바꿔 → "15.2°"
temp = soup.select_one(".temperature_text").get_text(strip=True).replace("현재 온도", "")
print("현재 온도:", temp)

# 2) 어제보다 ~ / 맑음
#   p.summary : <p> 태그 중 class="summary" 인 것
#   get_text(" ", strip=True) : 들쭉날쭉한 원래 공백을 다 지우고(strip) 조각 사이를 한 칸(" ")씩 다시 띄움
#     → "어제보다 4.4° 높아요 맑음"  (" " 없으면 "어제보다4.4°높아요맑음")
summary = soup.select_one("p.summary").get_text(" ", strip=True)
print("요약:", summary)

# 3) 체감·습도·바람 — <div class="sort"> 하나에 이름(dt) + 값(dd)이 한 쌍씩 들어 있음
#   HTML 구조:  <dl class="summary_list">
#                 <div class="sort"> <dt class="term">체감</dt> <dd class="desc">15.2°</dd> </div>
#                 <div class="sort"> <dt class="term">습도</dt> <dd class="desc">46%</dd>   </div>
#                 ...
#   "dl.summary_list div.sort" : summary_list 안의 sort 상자들을 전부(select → 리스트) 가져와 하나씩 item 으로
for item in soup.select("dl.summary_list div.sort"):
    # soup(문서 전체)이 아니라 item(상자 하나) 안에서만 찾으니까, 이름과 값이 항상 짝이 맞음
    name = item.select_one("dt.term").get_text(strip=True)   # 체감
    value = item.select_one("dd.desc").get_text(strip=True)  # 15.2°
    print(name, ":", value)  # 쉼표로 나열하면 사이에 공백 한 칸씩 → "체감 : 15.2°"

# ⚠ 네이버가 class 이름을 바꾸면 select_one이 None을 돌려줘서
#   'NoneType' object has no attribute 'get_text' 에러가 남 → 개발자도구로 이름을 다시 확인할 것


현재 온도: 21.3°
요약: 어제보다 6.4° 높아요 맑음
체감 : 21.3°
습도 : 45%
남서풍 : 2.1m/s


In [31]:
# 날씨 영역 전체(<div class="weather_info">)를 통째로 가져오기

# ✗ 처음 시도 — None 이 나왔던 줄
#   res 는 노트북 앞쪽에서 만든 res = requests.get("http://naver.com") (네이버 "메인" 페이지)
#   메인 페이지에는 날씨 박스가 없으니 select_one 이 못 찾아서 None
bs_obj = BeautifulSoup(res.text,"html.parser")
weather = bs_obj.select_one("div.weather_info")

# ✓ 고친 줄 — 위 셀에서 날씨 검색 페이지를 담아 둔 soup 에서 찾기
#   같은 변수 weather 에 다시 넣으므로 바로 위 줄의 결과(None)는 덮어써져 사라짐
#   (그래서 위 두 줄은 지워도 결과가 같음 — 비교용으로 남겨 둔 것)
weather = soup.select_one("div.weather_info")

# 날씨 박스 안의 HTML 전체가 출력됨 (온도·요약·체감/습도/바람 등이 모두 이 안에 들어 있음)
print(weather)


<div class="weather_info"> <div class="status_wrap"> <h3 class="blind">오늘의 날씨</h3> <div class="_today"> <div class="weather_graphic"> <div class="weather_main"> <i class="wt_icon ico_wt1"><span class="blind">맑음</span></i> </div> <div class="temperature_text"> <strong><span class="blind">현재 온도</span>21.3<span class="celsius">°</span></strong> </div> </div> <div class="temperature_info"> <p class="summary">어제보다 <span class="temperature up">6.4° <span class="blind">높아요</span> </span> <span class="weather before_slash">맑음</span> </p> <dl class="summary_list"> <div class="sort"> <dt class="term">체감</dt> <dd class="desc">21.3°</dd> </div> <div class="sort"> <dt class="term">습도</dt> <dd class="desc">45%</dd> </div> <div class="sort"> <dt class="term">남서풍</dt> <dd class="desc">2.1m/s</dd> </div> </dl> </div> </div> <div class="report_card_wrap"> <ul class="today_chart_list"> <li class="item_today level2"> <a class="box" href="?where=nexearch&amp;sm=tab_etc&amp;pkid=227&amp;qvt=0&amp;query=%EC%

In [32]:
# 체감·습도·바람을 find / find_all 방식으로 꺼내기 (앞 셀의 select 방식과 결과는 같음)
# weather : 앞 셀에서 찾아 둔 날씨 박스 전체 (<div class="weather_info">)

# weather 박스 안에서 class="summary_list" 인 <dl> 태그 하나 찾기
#   find("태그", {"속성": "값"}) = select_one("dl.summary_list") 와 같은 뜻 (못 찾으면 None)
dl = weather.find("dl",{"class":"summary_list"})

# dl 이 None 이 아닐 때만 실행 (못 찾았는데 dl.find_all 하면 에러가 나니까 미리 막음)
if dl:
    divs = dl.find_all("div")   # dl 안의 <div> 전부 → 리스트 [체감 상자, 습도 상자, 바람 상자]
    print(len(divs))            # 상자 개수 → 3
    print(type(dl))             # <class 'bs4.element.Tag'> : 태그 하나라는 뜻

    # 상자를 하나씩 꺼내서 이름(dt)과 값(dd)을 출력
    for div in divs:
        dt_tag = div.find("dt")              # 이름 태그 <dt>체감</dt>
        if dt_tag:                           # 있을 때만 (없으면 None → get_text 에러 방지)
            # end=" " : 줄바꿈 대신 공백으로 끝냄 → 다음 print(값)가 같은 줄에 이어서 나옴
            print(dt_tag.get_text(strip=True),end=" ")

        dd_tag = div.find("dd")              # 값 태그 <dd>18.9°</dd>
        if dd_tag:
            print(dd_tag.get_text(strip=True))   # end 를 안 쓰면 기본값이 줄바꿈 → 다음 상자는 새 줄에서 시작
            # 결과: "체감 18.9°" 한 줄


else:
    # dl 을 못 찾은 경우 (네이버가 구조를 바꿨거나 weather 가 None 인 경우 등)
    print("날씨 정보를 찾을 수 없습니다.")

3
<class 'bs4.element.Tag'>
체감 21.3°
습도 45%
남서풍 2.1m/s


## 07. 네이버 증권 — 화면에 숫자가 없는 페이지 (API로 가져오기)

삼성전자(005930)의 **전일·시가·고가·저가·거래량·거래대금**을 가져옵니다.

| | 네이버 날씨 | 네이버 증권 |
|---|---|---|
| `requests`로 받은 HTML | 숫자가 **들어 있음** | **빈 틀만** 있음 (숫자 0개) |
| 숫자가 오는 곳 | HTML 안 | 자바스크립트가 따로 부르는 **데이터 주소(API)** |
| 꺼내는 방법 | `BeautifulSoup` + `select_one` | `res.json()` → 딕셔너리 |

**API 찾는 법:** F12 → **Network** 탭 → 새로고침 → 주소에 `api`가 들어간 요청을 클릭 → **Response**에서 숫자가 보이면 그 주소를 사용

> ⚠ 네이버 증권은 robots.txt로 자동 수집을 막아 두었습니다(`Disallow: /`). 실습으로 가끔 실행하는 정도만 하고, 반복·대량 수집은 공공데이터포털 API를 사용하세요.

In [33]:
import requests

# ===== 네이버 증권 — 삼성전자 시세 가져오기 =====
# stock.naver.com 페이지는 HTML에 숫자가 없음 (빈 틀만 오고, 숫자는 자바스크립트가 나중에 불러옴)
#   → BeautifulSoup으로 찾으면 None. 대신 그 숫자를 보내 주는 "데이터 주소(API)"에 직접 요청
#   → 찾는 법: F12 → Network 탭 → 새로고침 → 주소에 api 가 들어간 요청 확인

code = "005930"   # 종목코드 (삼성전자)
url = f"https://m.stock.naver.com/api/stock/{code}/integration"   # 화면 뒤에서 숫자를 보내 주는 데이터 주소
#   f-string : {code} 자리에 "005930" 이 들어감 → 종목코드만 바꾸면 다른 종목도 가능
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/155.0.0.0 Safari/537.36"}

res = requests.get(url, headers=headers, timeout=10)
data = res.json()   # JSON 글자 → 파이썬 딕셔너리 (HTML이 아니라서 BeautifulSoup 필요 없음)
#   data 모양(일부):
#   {"stockName": "삼성전자",
#    "totalInfos": [{"key": "전일", "value": "214,500"},
#                   {"key": "시가", "value": "207,000"}, ...]}

# 딕셔너리["키"] 로 값 꺼내기 → data["stockName"] = "삼성전자"
print(f"종목명: {data['stockName']} ({code})")
print("-" * 30)   # 구분선 ("-" 30개)

# totalInfos : [{'key': '전일', 'value': '214,500'}, {'key': '시가', ...}, ...] 형태의 리스트
#   (PER, 시총, 52주 최고 등 18개 항목이 들어 있음 → 그중 원하는 것만 골라 출력)
wanted = ["전일", "고가", "거래량", "시가", "저가", "대금"]   # 원하는 항목만 (화면의 '거래대금'은 여기선 '대금')
for info in data["totalInfos"]:       # info 하나 = {"key": "전일", "value": "214,500"}
    if info["key"] in wanted:         # 항목 이름이 wanted 리스트 안에 있으면
        print(f"{info['key']}: {info['value']}")   # "전일: 214,500"

# ⚠ 네이버 증권 robots.txt 는 자동 수집을 막아 둠(Disallow: /)
#   → 실습으로 가끔 한 번 실행하는 정도만. 반복문으로 여러 종목·자동 반복 수집은 하지 말 것
#   → 주가를 제대로 모으려면 공공데이터포털 "금융위원회_주식시세정보" API (공식 경로)

종목명: 삼성전자 (005930)
------------------------------
전일: 214,500
시가: 207,000
고가: 215,500
저가: 203,500
거래량: 25,059,542
대금: 5조 2,533억


## 04.Selenium(동적 데이터 수집)

In [34]:
#!pip install selenium
# selenium : 웹 브라우저 자동화 라이브러리
# webdriver : 브라우저를 실행하하고 제어하는 기능을 제공
#   requests 는 HTML만 받아오지만, selenium 은 "진짜 크롬 창"을 띄워서 사람처럼 조종함
#   → 자바스크립트가 나중에 그려 주는 내용(동적 데이터)도 화면에 뜬 그대로 가져올 수 있음
from selenium import webdriver

# ChromeDriver 서비스 가져오기
# ChromeDriver는 Selenium의 명령을 Chrome 브라우저에 전달하는 역할
#   (파이썬 코드 → ChromeDriver(통역사) → 크롬 브라우저)
from selenium.webdriver.chrome.service import Service
import time   # time.sleep(초) : 페이지가 뜰 때까지 기다리는 데 사용

service = Service() # Selenium 서비스 생성   (ChromeDriver 경로를 비워 두면 자동으로 찾아/내려받아 줌)
options = webdriver.ChromeOptions() # Chrome 브라우저 옵션 설정   (창 크기, 화면 없이 실행 등. 지금은 기본값)

# Chrome 브라우저 실행 객체 → 이 줄에서 새 크롬 창이 뜸
#   driver = 크롬 창을 조종하는 "리모컨". 창을 닫거나 driver.quit() 하면 리모컨이 끊겨서
#   이후 명령에서 InvalidSessionIdException 이 남 → 이 셀을 다시 실행하면 해결
driver = webdriver.Chrome(service=service, options=options)

url = "http://naver.com"
driver.get(url)   # 크롬 창에서 이 주소로 이동 (주소창에 입력하고 Enter 친 것과 같음)


* 키 제어

In [35]:
# (위 셀과 같은 내용 — 키 제어 실습을 위해 크롬 창을 새로 띄움)
# selenium : 웹 브라우저 자동화 라이브러리
# webdriver : 브라우저를 실행하하고 제어하는 기능을 제공
#   requests 는 HTML만 받아오지만, selenium 은 "진짜 크롬 창"을 띄워서 사람처럼 조종함
#   → 자바스크립트가 나중에 그려 주는 내용(동적 데이터)도 화면에 뜬 그대로 가져올 수 있음
from selenium import webdriver

# ChromeDriver 서비스 가져오기
# ChromeDriver는 Selenium의 명령을 Chrome 브라우저에 전달하는 역할
#   (파이썬 코드 → ChromeDriver(통역사) → 크롬 브라우저)
from selenium.webdriver.chrome.service import Service
import time   # time.sleep(초) : 페이지가 뜰 때까지 기다리는 데 사용

service = Service() # Selenium 서비스 생성   (ChromeDriver 경로를 비워 두면 자동으로 찾아/내려받아 줌)
options = webdriver.ChromeOptions() # Chrome 브라우저 옵션 설정   (창 크기, 화면 없이 실행 등. 지금은 기본값)

# Chrome 브라우저 실행 객체 → 이 줄에서 새 크롬 창이 뜸
#   driver = 크롬 창을 조종하는 "리모컨". 창을 닫거나 driver.quit() 하면 리모컨이 끊겨서
#   이후 명령에서 InvalidSessionIdException 이 남 → 이 셀을 다시 실행하면 해결
driver = webdriver.Chrome(service=service, options=options)

url = "http://naver.com"
driver.get(url)   # 크롬 창에서 네이버로 이동

In [36]:
from selenium.webdriver.common.by import By     # By : "무엇으로 찾을지" (ID, CSS_SELECTOR, CLASS_NAME 등)
from selenium.webdriver.common.keys import Keys  # Keys : 키보드 특수키 (ENTER, TAB, ESC, 방향키 등)

# 네이버 검색창 찾기 — 개발자도구로 보면 <input id="query" ...> 이라서 By.ID, "query"
element = driver.find_element(By.ID, "query")
element.send_keys("컴퓨터")       # 검색창에 "컴퓨터" 라고 키보드로 입력
element.send_keys(Keys.ENTER)    # Enter 키 누르기 → 검색 실행


* MiniPrj : 다음 영화 역대 관객 순위 TOP5(5년치)

In [ ]:
# ===== MiniPrj 준비: 크롬을 띄우고 다음(Daum) "2026년영화순위" 검색 페이지 열기 =====
# selenium : 웹 브라우저 자동화 라이브러리
# webdriver : 브라우저를 실행하하고 제어하는 기능을 제공
#   requests 는 HTML만 받아오지만, selenium 은 "진짜 크롬 창"을 띄워서 사람처럼 조종함
#   → 자바스크립트가 나중에 그려 주는 내용(동적 데이터)도 화면에 뜬 그대로 가져올 수 있음
from selenium import webdriver

# ChromeDriver 서비스 가져오기
# ChromeDriver는 Selenium의 명령을 Chrome 브라우저에 전달하는 역할
#   (파이썬 코드 → ChromeDriver(통역사) → 크롬 브라우저)
from selenium.webdriver.chrome.service import Service
import time   # time.sleep(초) : 페이지가 뜰 때까지 기다리는 데 사용

service = Service() # Selenium 서비스 생성   (ChromeDriver 경로를 비워 두면 자동으로 찾아/내려받아 줌)
options = webdriver.ChromeOptions() # Chrome 브라우저 옵션 설정   (창 크기, 화면 없이 실행 등. 지금은 기본값)

# Chrome 브라우저 실행 객체 → 이 줄에서 새 크롬 창이 뜸
#   driver = 크롬 창을 조종하는 "리모컨". 창을 닫거나 driver.quit() 하면 리모컨이 끊겨서
#   이후 명령에서 InvalidSessionIdException 이 남 → 이 셀을 다시 실행하면 해결
driver = webdriver.Chrome(service=service, options=options)

# try / except : try 안에서 에러가 나도 프로그램이 멈추지 않고 except 로 넘어감
try:
    # q= 뒤의 %EB%85%84... 는 "년영화순위"를 주소용 글자로 바꾼(인코딩한) 것 → "2026년영화순위" 검색
    url = "https://search.daum.net/search?w=tot&q=2026%EB%85%84%EC%98%81%ED%99%94%EC%88%9C%EC%9C%84&DA=MOR&rtmaxcoll=MOR"
    driver.get(url)
    time.sleep(3)   # 자바스크립트가 영화 목록을 다 그릴 때까지 3초 기다림
except Exception as e:
    # Exception : 거의 모든 에러를 잡는 그물 / e : 에러 쪽지 (Selenium·크롬이 써서 넘겨준 메시지)
    print(e)

* 5년 영화 데이터 수집

In [ ]:
import requests                              # 포스터 이미지를 받아올 때 사용
from IPython.display import display, Image   # 노트북 출력 칸에 이미지를 보여 주는 도구
# ← 추가
# 2022 ~ 2026년을 차례로 돌면서, 연도별 영화 순위 TOP5의 제목·누적 관객·포스터를 수집
for year in range(2022, 2027): # ← 2022 ~ 2026   (range 는 끝 숫자 2027 을 포함하지 않음 → 5번 반복)
    try:
        # f-string : {year} 자리에 2022, 2023 … 이 들어감 → 연도만 바뀐 검색 주소 (한글은 크롬이 알아서 인코딩)
        url = f"https://search.daum.net/search?w=tot&q={year}년영화순위&DA=MOR&rtmaxcoll=MOR"   # ← 추가: 연도만 바꾼 주소
        driver.get(url)        # ← 그 연도 페이지로 이동
        time.sleep(3)          # ← 페이지 뜰 때까지 기다리기 (+ 서버에 너무 빠르게 연속 요청하지 않기)
        print(f"===== {year}년 =====")
    except Exception as e:
        print(e)

        # ⚠ 들여쓰기 주의: 아래 줄들이 전부 except 블록 "안"에 들어가 있음
        #   → 에러가 났을 때만 실행되고, 정상일 때는 영화 수집을 하나도 안 함
        #   → 의도대로라면 movies = ... 부터 끝까지를 한 단계 내어써서 try/except 와 같은 높이(for year 바로 안)에 두어야 함

        # "실시간 예매율" 이름표가 붙은 c-list-doc 상자 → 그 안의 li(영화 한 편) 전부
        #   [data-title="..."] : 속성 값으로 찾기 / 띄어쓰기 + li : "그 안에 있는" li
        movies = driver.find_elements(By.CSS_SELECTOR, 'c-list-doc[data-title="실시간 예매율"] li')

        # enumerate : 0, 1, 2 … 번호(idx)와 영화(movie)를 함께 꺼냄
        for idx, movie in enumerate(movies):
            if idx == 5:   # 0~4 → 5편까지만, 6번째(idx 5)가 되면 반복 중단
                break

            # movie(영화 한 편) 안에서만 찾으니까 제목·관객수·포스터가 항상 짝이 맞음
            title = movie.find_element(By.CSS_SELECTOR, ".tit-g").text           # 제목  <strong class="tit-g">
            audience = movie.find_element(By.CSS_SELECTOR, ".conts-desc").text   # 누적 관객 <p class="conts-desc">

            # 포스터 주소 — 이미지는 글자가 없어서 .text 가 아니라 src "속성"을 꺼냄
            img_url = movie.find_element(By.CSS_SELECTOR, "img").get_attribute("src")
            # 주소 속 ".fwebp" = "webp 형식으로 달라"는 표시 → 지우면 서버가 jpg 로 보내 줌
            img_url = img_url.replace(".fwebp","")
            res = requests.get(img_url, timeout=10)   # 그 주소로 이미지 받기
            # "wb" : 바이트 그대로 쓰기 / 파일 이름 예: 2022_1.jpg, 2022_2.jpg … 2026_5.jpg
            with open(f"{year}_{idx + 1}.jpg", "wb") as f:
                f.write(res.content)   # .text(글자)가 아니라 .content(그림 데이터)

            print(title,"->", audience)
            display(Image(url=img_url, width=120))                     # ← 추가: 노트북에 포스터 표시


===== 2022년 =====
범죄도시2 -> 누적 1,269만명


탑건: 매버릭 -> 누적 817만명


아바타: 물의 길 -> 누적 731만명


한산: 용의 출현 -> 누적 726만명


공조2: 인터내셔날 -> 누적 698만명


...

===== 2026년 =====
왕과 사는 남자 -> 누적 1,693만명


오디세이 -> 누적 1,229만명


스파이더맨: 브랜드 뉴 데이 -> 누적 905만명


군체 -> 누적 595만명


호프 -> 누적 459만명
